### Testing different codes

In [3]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV

# ==========================================
# 1. Load the Original Data
# ==========================================
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

# Extract targets and drop IDs
y_train = original_df['Revenue']
X_train_raw = original_df.drop(columns=['Revenue', 'Session_ID'], errors='ignore')
X_test_raw = test_df.drop(columns=['Session_ID'], errors='ignore')

# ==========================================
# 2. Impute Missing Values Safely with Pandas
# ==========================================
# Identify column categories
numeric_features = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

# Fit medians and modes exclusively on training set
train_medians = X_train_raw[numeric_features].median()
train_modes = X_train_raw[categorical_features].mode().iloc[0]

# Apply Imputation
X_train_imp = X_train_raw.copy()
X_train_imp[numeric_features] = X_train_imp[numeric_features].fillna(train_medians)
X_train_imp[categorical_features] = X_train_imp[categorical_features].fillna(train_modes)

X_test_imp = X_test_raw.copy()
X_test_imp[numeric_features] = X_test_imp[numeric_features].fillna(train_medians)
X_test_imp[categorical_features] = X_test_imp[categorical_features].fillna(train_modes)

# ==========================================
# 3. Domain Feature Engineering
# ==========================================
def engineer_features(df):
    df = df.copy()
    
    # 1. High-impact binary indicator for PageValues
    if 'PageValues' in df.columns:
        df['Has_PageValue'] = (df['PageValues'] > 0).astype(int)
    
    # 2. Aggregate counts & durations
    page_cols = [c for c in ['Administrative', 'Informational', 'ProductRelated'] if c in df.columns]
    dur_cols = [c for c in ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration'] if c in df.columns]
    
    if page_cols:
        df['Total_Pages'] = df[page_cols].sum(axis=1)
    if dur_cols:
        df['Total_Duration'] = df[dur_cols].sum(axis=1)
    
    # 3. Log transform highly skewed numeric columns
    num_cols = df.select_dtypes(include=['int64', 'float64']).columns
    for c in num_cols:
        df[c] = np.log1p(np.maximum(0, df[c]))
        
    return df

X_train_fe = engineer_features(X_train_imp)
X_test_fe = engineer_features(X_test_imp)

# Update column types after feature engineering
new_numeric_features = X_train_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()
new_categorical_features = X_train_fe.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

# ==========================================
# 4. Scaling & Encoding
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), new_numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), new_categorical_features)
    ])

X_train_processed_array = preprocessor.fit_transform(X_train_fe)
feature_names = preprocessor.get_feature_names_out()

# Reconstruct processed_df to meet submission requirements
processed_df = pd.DataFrame(X_train_processed_array, columns=feature_names)
processed_df['Revenue'] = y_train.values

# Prepare data arrays for training
X_processed = processed_df.drop(columns=['Revenue'])
y_processed = processed_df['Revenue']

# ==========================================
# 5. Model Training & Hyperparameter Tuning
# ==========================================
# Using default weights (class_weight=None) maximizes raw accuracy metric
param_grid = {
    'C': [0.01, 0.1, 1.0, 10.0, 50.0],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

base_model = LogisticRegression(max_iter=2000, random_state=42)
grid_search = GridSearchCV(base_model, param_grid, cv=5, scoring='accuracy', n_jobs=-1)
grid_search.fit(X_processed, y_processed)

# Assign best model to the required variable
model = grid_search.best_estimator_
print(f"Best CV Accuracy Score: {grid_search.best_score_:.4f}")
print(f"Best Parameters: {grid_search.best_params_}")

# ==========================================
# 6. Generate Predictions on Test Data
# ==========================================
X_test_processed_array = preprocessor.transform(X_test_fe)
X_test_processed = pd.DataFrame(X_test_processed_array, columns=feature_names)

predictions = model.predict(X_test_processed)

# ==========================================
# 7. Competition Submission Script
# ==========================================
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv("submission1.csv", index=False)

print("\n--- SUCCESS ---")
print("submission.csv created successfully.")
print("\nCheckpoints Summary:")
print(checkpoints)

Best CV Accuracy Score: 0.8839
Best Parameters: {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}

--- SUCCESS ---
submission.csv created successfully.

Checkpoints Summary:
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  33
6  row_retained_percent               100.0
7            model_name  LogisticRegression


c:\Users\daksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(


In [4]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# ==========================================
# 1. Load the Original Data
# ==========================================
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

y_train = original_df['Revenue']
X_train_raw = original_df.drop(columns=['Revenue', 'Session_ID'], errors='ignore')
X_test_raw = test_df.drop(columns=['Session_ID'], errors='ignore')

# ==========================================
# 2. Impute Missing Values Safely with Pandas
# ==========================================
numeric_features = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

train_medians = X_train_raw[numeric_features].median()
train_modes = X_train_raw[categorical_features].mode().iloc[0]

X_train_imp = X_train_raw.copy()
X_train_imp[numeric_features] = X_train_imp[numeric_features].fillna(train_medians)
X_train_imp[categorical_features] = X_train_imp[categorical_features].fillna(train_modes)

X_test_imp = X_test_raw.copy()
X_test_imp[numeric_features] = X_test_imp[numeric_features].fillna(train_medians)
X_test_imp[categorical_features] = X_test_imp[categorical_features].fillna(train_modes)

# ==========================================
# 3. Advanced Non-Linear Feature Engineering
# ==========================================
def engineer_advanced_features(df):
    df = df.copy()
    
    # 1. Key domain binary flags
    if 'PageValues' in df.columns:
        df['Has_PageValue'] = (df['PageValues'] > 0).astype(int)
        
    if 'BounceRates' in df.columns:
        df['Zero_Bounce'] = (df['BounceRates'] == 0).astype(int)
        
    # 2. Key interactions & non-linear ratios
    eps = 1e-6
    if 'PageValues' in df.columns and 'ExitRates' in df.columns:
        df['PV_x_Exit'] = df['PageValues'] * (1.0 - df['ExitRates'])
        
    if 'PageValues' in df.columns and 'BounceRates' in df.columns:
        df['PV_div_Bounce'] = df['PageValues'] / (df['BounceRates'] + eps)
        
    if 'ProductRelated' in df.columns and 'ProductRelated_Duration' in df.columns:
        df['Avg_Duration_Per_Product'] = df['ProductRelated_Duration'] / (df['ProductRelated'] + eps)
        
    # 3. Totals
    page_cols = [c for c in ['Administrative', 'Informational', 'ProductRelated'] if c in df.columns]
    dur_cols = [c for c in ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration'] if c in df.columns]
    
    if page_cols:
        df['Total_Pages'] = df[page_cols].sum(axis=1)
    if dur_cols:
        df['Total_Duration'] = df[dur_cols].sum(axis=1)
        
    return df

X_train_fe = engineer_advanced_features(X_train_imp)
X_test_fe = engineer_advanced_features(X_test_imp)

new_numeric_features = X_train_fe.select_dtypes(include=['int64', 'float64']).columns.tolist()
new_categorical_features = X_train_fe.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

# ==========================================
# 4. Power Transformation & Scaling
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        ('num', PowerTransformer(method='yeo-johnson'), new_numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), new_categorical_features)
    ])

X_train_processed_array = preprocessor.fit_transform(X_train_fe)
feature_names = preprocessor.get_feature_names_out()

# Reconstruct processed_df for submission requirements
processed_df = pd.DataFrame(X_train_processed_array, columns=feature_names)
processed_df['Revenue'] = y_train.values

X_processed = processed_df.drop(columns=['Revenue'])
y_processed = processed_df['Revenue']

# ==========================================
# 5. Model Training & Threshold Optimization
# ==========================================
# Train fine-tuned Logistic Regression
model = LogisticRegression(C=2.0, max_iter=3000, random_state=42)
model.fit(X_processed, y_processed)

# Cross-Validation to find the optimal decision threshold for accuracy
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
thresholds = np.linspace(0.35, 0.65, 31)
threshold_scores = {t: [] for t in thresholds}

# Get out-of-fold probability predictions
cv_probs = np.zeros(len(X_processed))
for train_idx, val_idx in skf.split(X_processed, y_processed):
    X_tr, X_val = X_processed.iloc[train_idx], X_processed.iloc[val_idx]
    y_tr, y_val = y_processed.iloc[train_idx], y_processed.iloc[val_idx]
    
    fold_model = LogisticRegression(C=2.0, max_iter=3000, random_state=42)
    fold_model.fit(X_tr, y_tr)
    probs_val = fold_model.predict_proba(X_val)[:, 1]
    
    for t in thresholds:
        preds = probs_val >= t
        threshold_scores[t].append(accuracy_score(y_val, preds))

avg_scores = {t: np.mean(scores) for t, scores in threshold_scores.items()}
best_threshold = max(avg_scores, key=avg_scores.get)

print(f"Optimal Classification Threshold for Accuracy: {best_threshold:.3f}")
print(f"Estimated Cross-Validation Accuracy: {avg_scores[best_threshold]:.4f}")

# ==========================================
# 6. Generate Predictions on Test Data
# ==========================================
X_test_processed_array = preprocessor.transform(X_test_fe)
X_test_processed = pd.DataFrame(X_test_processed_array, columns=feature_names)

# Apply probabilities with the optimal threshold
test_probabilities = model.predict_proba(X_test_processed)[:, 1]
predictions = (test_probabilities >= best_threshold)

# ==========================================
# 7. Competition Submission Script
# ==========================================
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv("submission2.csv", index=False)

print("\n--- SUCCESS ---")
print("submission.csv updated successfully with threshold-optimized predictions.")
print("\nCheckpoints Summary:")
print(checkpoints)

Optimal Classification Threshold for Accuracy: 0.520
Estimated Cross-Validation Accuracy: 0.8847

--- SUCCESS ---
submission.csv updated successfully with threshold-optimized predictions.

Checkpoints Summary:
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  37
6  row_retained_percent               100.0
7            model_name  LogisticRegression


In [11]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PowerTransformer, SplineTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# ==========================================
# 1. Load the Original Data
# ==========================================
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

y_train = original_df['Revenue']
X_train_raw = original_df.drop(columns=['Revenue', 'Session_ID'], errors='ignore')
X_test_raw = test_df.drop(columns=['Session_ID'], errors='ignore')

# ==========================================
# 2. Impute Missing Values Safely with Pandas
# ==========================================
numeric_features = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

train_medians = X_train_raw[numeric_features].median()
train_modes = X_train_raw[categorical_features].mode().iloc[0]

X_train_imp = X_train_raw.copy()
X_train_imp[numeric_features] = X_train_imp[numeric_features].fillna(train_medians)
X_train_imp[categorical_features] = X_train_imp[categorical_features].fillna(train_modes)

X_test_imp = X_test_raw.copy()
X_test_imp[numeric_features] = X_test_imp[numeric_features].fillna(train_medians)
X_test_imp[categorical_features] = X_test_imp[categorical_features].fillna(train_modes)

# ==========================================
# 3. Maximum Domain Feature Engineering
# ==========================================
def engineer_ultra_features(df_train, df_test):
    df_tr = df_train.copy()
    df_te = df_test.copy()
    eps = 1e-6

    for df in [df_tr, df_te]:
        # 1. Non-linear Bins & Binary Flags
        if 'PageValues' in df.columns:
            df['Has_PageValue'] = (df['PageValues'] > 0).astype(int)
            df['High_PageValue'] = (df['PageValues'] > 20).astype(int)
            
        if 'BounceRates' in df.columns:
            df['Zero_Bounce'] = (df['BounceRates'] == 0).astype(int)
            
        if 'ExitRates' in df.columns:
            df['Low_Exit'] = (df['ExitRates'] < 0.02).astype(int)

        # 2. Key Domain Ratios
        if 'PageValues' in df.columns and 'ExitRates' in df.columns:
            df['PV_x_1minusExit'] = df['PageValues'] * (1.0 - df['ExitRates'])
            
        if 'BounceRates' in df.columns and 'ExitRates' in df.columns:
            df['Bounce_to_Exit_Ratio'] = df['BounceRates'] / (df['ExitRates'] + eps)

        if 'ProductRelated' in df.columns and 'ProductRelated_Duration' in df.columns:
            df['Avg_Duration_Per_Product'] = df['ProductRelated_Duration'] / (df['ProductRelated'] + eps)

        # 3. Aggregate Session Metrics
        page_cols = [c for c in ['Administrative', 'Informational', 'ProductRelated'] if c in df.columns]
        dur_cols = [c for c in ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration'] if c in df.columns]

        if page_cols:
            df['Total_Pages'] = df[page_cols].sum(axis=1)
        if dur_cols:
            df['Total_Duration'] = df[dur_cols].sum(axis=1)
            
        if 'Total_Duration' in df.columns and 'Total_Pages' in df.columns:
            df['Page_Engagement_Density'] = df['Total_Duration'] / (df['Total_Pages'] + eps)

    # 4. Frequency Encoding for High-Cardinality / Categorical Features
    freq_cols = [c for c in ['OperatingSystems', 'Browser', 'Region', 'TrafficType', 'Month', 'VisitorType'] 
                 if c in df_tr.columns]
    
    for c in freq_cols:
        freq_map = df_tr[c].value_counts(normalize=True).to_dict()
        df_tr[f'{c}_freq'] = df_tr[c].map(freq_map).fillna(0)
        df_te[f'{c}_freq'] = df_te[c].map(freq_map).fillna(0)

    return df_tr, df_te

X_train_fe, X_test_fe = engineer_ultra_features(X_train_imp, X_test_imp)

# Separate PageValues for Spline Transformation to model non-linear score jumps
spline_col = ['PageValues'] if 'PageValues' in X_train_fe.columns else []
new_numeric_features = [c for c in X_train_fe.select_dtypes(include=['int64', 'float64']).columns if c not in spline_col]
new_categorical_features = X_train_fe.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

# ==========================================
# 4. Preprocessing & Spline Pipeline
# ==========================================
transformers_list = [
    ('num', PowerTransformer(method='yeo-johnson'), new_numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), new_categorical_features)
]

if spline_col:
    transformers_list.append(('spline', SplineTransformer(n_knots=5, degree=3), spline_col))

preprocessor = ColumnTransformer(transformers=transformers_list)

X_train_processed_array = preprocessor.fit_transform(X_train_fe)
feature_names = preprocessor.get_feature_names_out()

# Reconstruct processed_df for submission requirements
processed_df = pd.DataFrame(X_train_processed_array, columns=feature_names)
processed_df['Revenue'] = y_train.values

X_processed = processed_df.drop(columns=['Revenue'])
y_processed = processed_df['Revenue']

# ==========================================
# 5. 10-Fold Out-of-Fold Ensembling & Threshold Tuning
# ==========================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_probs = np.zeros(len(X_processed))
X_test_processed_array = preprocessor.transform(X_test_fe)
X_test_processed = pd.DataFrame(X_test_processed_array, columns=feature_names)
test_probs_folds = np.zeros(len(X_test_processed))

# Hyperparameter: C=1.5 with saga solver for optimal regularized fit
for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y_processed)):
    X_tr, X_val = X_processed.iloc[train_idx], X_processed.iloc[val_idx]
    y_tr, y_val = y_processed.iloc[train_idx], y_processed.iloc[val_idx]
    
    fold_model = LogisticRegression(C=1.5, solver='saga', max_iter=4000, random_state=42 + fold)
    fold_model.fit(X_tr, y_tr)
    
    oof_probs[val_idx] = fold_model.predict_proba(X_val)[:, 1]
    test_probs_folds += fold_model.predict_proba(X_test_processed)[:, 1] / 10.0

# Fine-grained threshold search for maximum accuracy score
thresholds = np.linspace(0.35, 0.65, 61)
best_threshold = 0.5
best_acc = 0.0

for t in thresholds:
    acc = accuracy_score(y_processed, oof_probs >= t)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"Optimal Classification Threshold: {best_threshold:.4f}")
print(f"10-Fold OOF Estimated Accuracy: {best_acc * 100:.3f}%")

# Train final single model instance to assign to the 'model' variable (Submission Check Requirement)
model = LogisticRegression(C=1.5, solver='saga', max_iter=4000, random_state=42)
model.fit(X_processed, y_processed)

# Final ensembled predictions using the optimal threshold
predictions = (test_probs_folds >= best_threshold)

# ==========================================
# 6. Final Competition Submission Code
# ==========================================
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv("submission3.csv", index=False)

print("\n--- SUCCESS ---")
print("submission.csv updated successfully with ultra-engineered ensembled predictions.")
print("\nCheckpoints Summary:")
print(checkpoints)

Optimal Classification Threshold: 0.4800
10-Fold OOF Estimated Accuracy: 88.625%

--- SUCCESS ---
submission.csv updated successfully with ultra-engineered ensembled predictions.

Checkpoints Summary:
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  52
6  row_retained_percent               100.0
7            model_name  LogisticRegression


In [12]:
import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, KBinsDiscretizer, PolynomialFeatures
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score

# ==========================================
# 1. Load Original Data
# ==========================================
original_df = pd.read_csv("train.csv")
test_df = pd.read_csv("test.csv")

y_train = original_df['Revenue']
X_train_raw = original_df.drop(columns=['Revenue', 'Session_ID'], errors='ignore')
X_test_raw = test_df.drop(columns=['Session_ID'], errors='ignore')

# ==========================================
# 2. Impute Missing Values Safely
# ==========================================
numeric_features = X_train_raw.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X_train_raw.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

train_medians = X_train_raw[numeric_features].median()
train_modes = X_train_raw[categorical_features].mode().iloc[0]

X_train_imp = X_train_raw.copy()
X_train_imp[numeric_features] = X_train_imp[numeric_features].fillna(train_medians)
X_train_imp[categorical_features] = X_train_imp[categorical_features].fillna(train_modes)

X_test_imp = X_test_raw.copy()
X_test_imp[numeric_features] = X_test_imp[numeric_features].fillna(train_medians)
X_test_imp[categorical_features] = X_test_imp[categorical_features].fillna(train_modes)

# ==========================================
# 3. Explicit Feature Transformations
# ==========================================
def create_step_features(df):
    df = df.copy()
    eps = 1e-6
    
    # 1. Critical Binary Indicators
    if 'PageValues' in df.columns:
        df['PV_is_zero'] = (df['PageValues'] == 0).astype(int)
        df['PV_gt_0'] = (df['PageValues'] > 0).astype(int)
        df['PV_gt_10'] = (df['PageValues'] > 10).astype(int)
        
    if 'BounceRates' in df.columns:
        df['Zero_Bounce'] = (df['BounceRates'] == 0).astype(int)
        
    if 'ExitRates' in df.columns:
        df['Low_Exit'] = (df['ExitRates'] < 0.02).astype(int)

    # 2. Aggregations & Non-linear Ratios
    page_cols = [c for c in ['Administrative', 'Informational', 'ProductRelated'] if c in df.columns]
    dur_cols = [c for c in ['Administrative_Duration', 'Informational_Duration', 'ProductRelated_Duration'] if c in df.columns]

    if page_cols:
        df['Total_Pages'] = df[page_cols].sum(axis=1)
    if dur_cols:
        df['Total_Duration'] = df[dur_cols].sum(axis=1)

    if 'PageValues' in df.columns and 'ExitRates' in df.columns:
        df['PV_per_Exit'] = df['PageValues'] / (df['ExitRates'] + eps)

    if 'Total_Duration' in df.columns and 'Total_Pages' in df.columns:
        df['Time_per_Page'] = df['Total_Duration'] / (df['Total_Pages'] + eps)

    return df

X_train_fe = create_step_features(X_train_imp)
X_test_fe = create_step_features(X_test_imp)

# Define column subgroups for target preprocessing
binned_cols = [c for c in ['PageValues', 'ExitRates', 'BounceRates', 'Total_Duration'] if c in X_train_fe.columns]
std_num_cols = [c for c in X_train_fe.select_dtypes(include=['int64', 'float64']).columns if c not in binned_cols]
cat_cols = X_train_fe.select_dtypes(include=['object', 'str', 'bool']).columns.tolist()

# ==========================================
# 4. Column Transformer Pipeline
# ==========================================
preprocessor = ColumnTransformer(
    transformers=[
        ('num_std', StandardScaler(), std_num_cols),
        ('num_bin', KBinsDiscretizer(n_bins=5, encode='onehot-dense', strategy='quantile'), binned_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), cat_cols)
    ])

X_train_processed_array = preprocessor.fit_transform(X_train_fe)
feature_names = preprocessor.get_feature_names_out()

# Reconstruct processed_df to fulfill submission checks
processed_df = pd.DataFrame(X_train_processed_array, columns=feature_names)
processed_df['Revenue'] = y_train.values

X_processed = processed_df.drop(columns=['Revenue'])
y_processed = processed_df['Revenue']

# ==========================================
# 5. Model Training & 10-Fold OOF Predictions
# ==========================================
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

oof_probs = np.zeros(len(X_processed))
X_test_processed_array = preprocessor.transform(X_test_fe)
X_test_processed = pd.DataFrame(X_test_processed_array, columns=feature_names)
test_probs_folds = np.zeros(len(X_test_processed))

# Fine-tuned L2 Logistic Regression with Optimized C Penalty
for fold, (train_idx, val_idx) in enumerate(skf.split(X_processed, y_processed)):
    X_tr, X_val = X_processed.iloc[train_idx], X_processed.iloc[val_idx]
    y_tr, y_val = y_processed.iloc[train_idx], y_processed.iloc[val_idx]
    
    fold_model = LogisticRegression(C=0.8, max_iter=3000, random_state=42 + fold)
    fold_model.fit(X_tr, y_tr)
    
    oof_probs[val_idx] = fold_model.predict_proba(X_val)[:, 1]
    test_probs_folds += fold_model.predict_proba(X_test_processed)[:, 1] / 10.0

# Precise Threshold Sweep for Pure Accuracy Maximization
thresholds = np.linspace(0.40, 0.60, 101)
best_threshold = 0.5
best_acc = 0.0

for t in thresholds:
    acc = accuracy_score(y_processed, oof_probs >= t)
    if acc > best_acc:
        best_acc = acc
        best_threshold = t

print(f"Optimal Threshold: {best_threshold:.4f}")
print(f"Estimated 10-Fold Cross-Validation Accuracy: {best_acc * 100:.3f}%")

# Primary model instance for checkpoint validation
model = LogisticRegression(C=0.8, max_iter=3000, random_state=42)
model.fit(X_processed, y_processed)

# Generate final predictions using ensembled probabilities and optimal threshold
predictions = (test_probs_folds >= best_threshold)

# ==========================================
# 6. Generate Official Submission File
# ==========================================
original_missing = original_df.isnull().sum().sum()
processed_missing = processed_df.isnull().sum().sum()

original_rows, original_columns = original_df.shape
processed_rows, processed_columns = processed_df.shape

row_retained_percent = (processed_rows / original_rows) * 100

final_model = model
if hasattr(final_model, "best_estimator_"):
    final_model = final_model.best_estimator_
if hasattr(final_model, "steps"):
    final_model = final_model.steps[-1][1]

model_name = final_model.__class__.__name__

checkpoints = pd.DataFrame({
    "id": [
        "original_missing",
        "processed_missing",
        "original_rows",
        "processed_rows",
        "original_columns",
        "processed_columns",
        "row_retained_percent",
        "model_name"
    ],
    "value": [
        original_missing,
        processed_missing,
        original_rows,
        processed_rows,
        original_columns,
        processed_columns,
        round(row_retained_percent, 2),
        model_name
    ]
})

prediction_output = pd.DataFrame({
    "id": test_df["Session_ID"].astype(str),
    "value": np.asarray(predictions).astype(str)
})

submission = pd.concat(
    [checkpoints, prediction_output],
    ignore_index=True
)

submission.to_csv("submission4.csv", index=False)

print("\n--- SUCCESS ---")
print("submission.csv created successfully.")
print("\nCheckpoints Summary:")
print(checkpoints)

c:\Users\daksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:304: FutureWarning: The current default behavior, quantile_method='linear', will be changed to quantile_method='averaged_inverted_cdf' in scikit-learn version 1.9 to naturally support sample weight equivalence properties by default. Pass quantile_method='averaged_inverted_cdf' explicitly to silence this warning.
  warnings.warn(
c:\Users\daksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 0 are removed. Consider decreasing the number of bins.
  warnings.warn(
c:\Users\daksh\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\preprocessing\_discretization.py:396: UserWarning: Bins whose width are too small (i.e., <= 1e-8) in feature 2 are removed. Consider decreasing the number of bins.
  warnings.warn(


Optimal Threshold: 0.5180
Estimated 10-Fold Cross-Validation Accuracy: 88.504%

--- SUCCESS ---
submission.csv created successfully.

Checkpoints Summary:
                     id               value
0      original_missing                2957
1     processed_missing                   0
2         original_rows                9864
3        processed_rows                9864
4      original_columns                  19
5     processed_columns                  50
6  row_retained_percent               100.0
7            model_name  LogisticRegression
